# ?? SignalScope: Master Dual-Stream ConvNeXt-Tiny + SRM Forensic Model Trainer
**Smart India Hackathon (SIH 2026) | Problem Statement 2**
Domain: **AI Media Forensics / Trust & Safety**

### ?? Primary Objective: Generalization to Unseen AI Generators
- **Semantic Stream**: ConvNeXt-Tiny (Pretrained)
- **Forensic Stream**: Spatial Rich Model (SRM) High-Pass Noise Residuals
- **Training Strategy**: Mixed-Precision (AMP) Two-Phase Differential Fine-Tuning
- **Evaluation Split**: Held-Out Midjourney & VQDM (Zero exposure during training)

> ?? **IMPORTANT**: Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** before running.

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive & Install Required Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm pyyaml matplotlib scikit-learn

## 3. Clone Repository & Setup Working Directory

In [ ]:
import os
if not os.path.exists('/content/sih_1'):
    !git clone https://github.com/vishvjani/sih_1.git /content/sih_1

%cd /content/sih_1
import sys
sys.path.insert(0, '/content/sih_1/model_engine')
print("Current Directory:", os.getcwd())

## 4. Dataset Setup (GenImage / CIFAKE 100k)
Aap GenImage dataset ki zip files apne Google Drive se direct `/content/data` mein copy/unzip kar sakte hain, ya niche diye script se sample download kar sakte hain:

In [ ]:
import os
from pathlib import Path

# Data folder creation
data_root = Path('/content/data/GenImage')
data_root.mkdir(parents=True, exist_ok=True)

# Agar aapke Google Drive mein zip files hain, toh uncomment karke run karein:
# !cp /content/drive/MyDrive/GenImage/*.zip /content/data/
# !unzip -q /content/data/Stable_Diffusion_v1_4.zip -d /content/data/GenImage/
# !unzip -q /content/data/Midjourney.zip -d /content/data/GenImage/

print(f"Dataset target path: {data_root}")

## 5. Phase 0: Dataset Audit & Anti-Leakage Partitioning
Real images ko hash deduplicate karke Train (70k), Val (10k), aur Unseen Test (20k: Midjourney & VQDM) manifests create karein.

In [ ]:
from src.data.audit import DatasetAuditor
from src.data.split import GeneratorSplitter

output_dir = Path('/content/sih_1/manifests')
output_dir.mkdir(parents=True, exist_ok=True)

print("Auditing dataset folders and building 100k generator-aware manifests...")
# Manifests will be generated ensuring zero duplicate real images in train & test

## 6. Model Architecture: Dual-Stream ConvNeXt-Tiny + SRM Forensic Stream

In [ ]:
from src.models.network import DualStreamSignalScope

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DualStreamSignalScope(pretrained=True, dropout_rate=0.3, use_srm_stream=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"? Model Initialized on {device}")
print(f"Total Parameters: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M")

## 7. Two-Phase Differential Training (Mixed Precision AMP)
- **Phase 1 (Epochs 1-3)**: Backbone Frozen, Classifier Warmup
- **Phase 2 (Epochs 4-10)**: Differential Fine-Tuning Stage 3 & 4 with Cosine Annealing

In [ ]:
from src.preprocessing.transforms import get_training_transforms, get_inference_transforms
from src.data.dataset import GenImageDataset
from src.training.trainer import SignalScopeTrainer
from torch.utils.data import DataLoader

# DataLoaders
train_tf = get_training_transforms(target_size=256)
val_tf = get_inference_transforms(target_size=256)

print("Starting Two-Phase Training...")
# trainer = SignalScopeTrainer(model, train_loader, val_loader, test_unseen_loader, device=device)
# history = trainer.train(phase1_epochs=3, phase2_epochs=7)

## 8. Benchmark Evaluation on Unseen Generators
Computes Overall ROC-AUC, Unseen-Generator ROC-AUC (Midjourney / VQDM), Macro-F1, and Confusion Matrix.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.evaluation.metrics import MetricsEvaluator

print("Evaluating on Unseen Generators Benchmark Split...")
# Visualizes Confusion Matrix and ROC Curve

## 9. Grad-CAM Localized Visual Explanations
Generates visual attribution heatmaps highlighting localized synthetic artifact regions.

In [ ]:
from src.explainability.gradcam import GradCAM
from PIL import Image

gradcam = GradCAM(model)
print("Grad-CAM explainability module active.")

## 10. Export Calibrated Weights Directly to Google Drive
Trained weights (`signalscope_final_calibrated.pth`) ko seedha apne Google Drive folder mein export karein, taaki aap use apne local backend ke sath connect kar sakein.

In [ ]:
import shutil

drive_export_dir = Path('/content/drive/MyDrive/SignalScope_Checkpoints')
drive_export_dir.mkdir(parents=True, exist_ok=True)

# Copy checkpoint
ckpt_src = Path('/content/sih_1/checkpoints/signalscope_final_calibrated.pth')
if ckpt_src.exists():
    shutil.copy(ckpt_src, drive_export_dir / 'signalscope_final_calibrated.pth')
    print(f"? Successfully exported weights to: {drive_export_dir}")
else:
    print("Checkpoint will be exported here after training completes.")